In [ ]:
from pathlib import Path

import cdsapi

# Set True/False to skip the prompt; None asks each run.
DOWNLOAD_LATEST = None
LOCAL_ZIP = Path("era5-timeseries.zip")


def latest_local_zip() -> Path:
    if LOCAL_ZIP.exists():
        return LOCAL_ZIP
    zips = sorted(Path(".").glob("*.zip"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not zips:
        raise FileNotFoundError(
            "No local ERA5 zip found. Re-run this cell and answer yes to download."
        )
    return zips[0]


if DOWNLOAD_LATEST is None:
    answer = input("Download latest ERA5 timeseries from CDS? [y/N] ").strip().lower()
    DOWNLOAD_LATEST = answer in {"y", "yes"}

if DOWNLOAD_LATEST:
    dataset = "reanalysis-era5-single-levels-timeseries"
    request = {
        "variable": [
            "10m_u_component_of_wind",
            "10m_v_component_of_wind",
            "10m_wind_gust_since_previous_post_processing",
            "significant_height_of_combined_wind_waves_and_swell"
        ],
        "location": {"longitude": -89.551, "latitude": 28.662},
        "date": ["1940-01-01/2026-09-11"],
        "data_format": "netcdf"
    }

    client = cdsapi.Client()
    wavezip = client.retrieve(dataset, request).download(str(LOCAL_ZIP))
else:
    wavezip = str(latest_local_zip())

print(f"Using {wavezip}")


In [ ]:
import re
import xarray as xr
from zipfile import ZipFile

zipfile = wavezip


if zipfile.endswith(".zip"):
    zipfile = ZipFile(zipfile, "r")
else:
    zipfile = None

if zipfile:
    wavenc = zipfile.extractall("./tmp")

    for file in zipfile.namelist():
        if re.match(r"^.+wav.+\.nc$", file):
            waveds = xr.open_dataset("./tmp/"+file, engine="netcdf4")

        if re.match(r"^.+sfc.+\.nc$", file):
            sfcds = xr.open_dataset("./tmp/"+file, engine="netcdf4")


In [ ]:
df = (
    waveds.to_dataframe()[["swh"]]
    .join(sfcds.to_dataframe()[["u10", "v10", "fg10"]])
)
df["s10"] = (df["u10"] ** 2 + df["v10"] ** 2) ** 0.5 * 1.943844
df["fg10"] = df["fg10"].astype("float") * 1.943844
df["swh"] = df["swh"].astype("float") * 3.28084
df


In [ ]:
import pandas as pd
from IPython.display import Markdown, display

hours = df[["swh", "s10"]].dropna()

if isinstance(hours.index, pd.MultiIndex):
    time_level = "time" if "time" in hours.index.names else hours.index.names[0]
    timestamps = pd.to_datetime(hours.index.get_level_values(time_level))
else:
    timestamps = pd.to_datetime(hours.index)

hours = hours.copy()
hours.index = timestamps

wave_thresholds = list(range(0, 17, 2))  # 0-16 ft, 2 ft increments
wind_thresholds = list(range(0, 36, 5))  # 0-35 kt, 5 kt increments
month_names = [pd.Timestamp(2024, month, 1).strftime("%b") for month in range(1, 13)]

monthly_exceedance = {}
for month, name in enumerate(month_names, start=1):
    month_hours = hours[hours.index.month == month]
    table = pd.DataFrame(
        {
            wind: [
                ((month_hours["swh"] > wave) | (month_hours["s10"] > wind)).mean() * 100
                for wave in wave_thresholds
            ]
            for wind in wind_thresholds
        },
        index=wave_thresholds,
    )
    table.index.name = "Wave height (ft)"
    table.columns.name = "Wind (kt)"
    monthly_exceedance[name] = table

    display(Markdown(f"### {name}"))
    display(table.round(2).map(lambda value: f"{value:.2f}%"))


In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(4, 3, figsize=(22, 22))
figure.suptitle(
    "Monthly exceedance: significant wave height OR 10 m wind",
    fontsize=16,
    fontweight="bold",
    y=0.995,
)

for axis, (name, table) in zip(axes.ravel(), monthly_exceedance.items()):
    axis.axis("off")
    axis.set_title(name, fontsize=13, fontweight="bold", pad=10)

    export_table = table.round(2).map(lambda value: f"{value:.2f}%")
    export_table.columns = [f"{wind} kt" for wind in export_table.columns]
    export_table = export_table.reset_index()

    plotted = axis.table(
        cellText=export_table.values,
        colLabels=export_table.columns,
        cellLoc="center",
        colLoc="center",
        loc="center",
        colWidths=[0.16] + [0.105] * table.shape[1],
    )
    plotted.auto_set_font_size(False)
    plotted.set_fontsize(7)
    plotted.scale(1, 1.45)

    for (row, column), cell in plotted.get_celld().items():
        cell.set_edgecolor("#B8C2CC")
        if row == 0:
            cell.set_facecolor("#2F5597")
            cell.set_text_props(color="white", weight="bold")
        elif column == 0:
            cell.set_facecolor("#2F5597")
            cell.set_text_props(color="white", weight="bold")
        elif row % 2 == 0:
            cell.set_facecolor("#EAF1F8")

figure.tight_layout(rect=[0, 0, 1, 0.98])
figure.savefig("monthly_wave_wind_exceedance.png", dpi=200, bbox_inches="tight")
plt.show()
